# Week 5 Integration Assignment — Full EDA Pipeline

This notebook performs a complete exploratory data analysis pipeline on a newly generated orders dataset.

The workflow follows the sequence:

**Generate → Diagnose → Clean → Verify → Visualize → Summarize**

The dataset is intentionally generated with several data-quality problems. These problems will first be diagnosed without modification, then addressed individually with justified cleaning decisions.

## 1. Dataset Generation

The orders dataset is generated from the exact specification provided for the integration assignment.

No cleaning or modification is performed during this stage. The purpose is to establish the raw dataset exactly as provided before beginning diagnosis.

In [3]:
import numpy as np
import pandas as pd

In [4]:
rng = np.random.default_rng(seed=42)
n = 5000

orders = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "order_date": pd.date_range("2024-01-01", periods=n, freq="h"),
    "customer_id": rng.integers(1000, 1200, size=n),
    "product_category": rng.choice(
        ["Electronics", "electronics", "Home Goods", "Apparel", "Books"], size=n
    ),
    "quantity": rng.integers(1, 8, size=n),
    "unit_price": rng.normal(45, 20, size=n).round(2),
    "region": rng.choice(
        ["North", "South", "East", "West", None],
        size=n,
        p=[0.24, 0.24, 0.24, 0.24, 0.04]
    ),
})

# Introduce the mess, on purpose — do not skip this part
orders.loc[
    rng.choice(n, 150, replace=False),
    "customer_id"
] = None

orders.loc[
    rng.choice(n, 30, replace=False),
    "quantity"
] *= -1

orders.loc[
    rng.choice(n, 20, replace=False),
    "unit_price"
] = 4999.99

orders = pd.concat([
    orders,
    orders.sample(15, random_state=1)
])

### Initial Dataset Verification

Before performing any diagnosis or cleaning, the shape and first few records of the generated dataset are inspected to confirm that the required dataset was created successfully.

In [5]:
print("Dataset shape:", orders.shape)

Dataset shape: (5015, 7)


In [6]:
orders.head()

,order_id,order_date,customer_id,product_category,quantity,unit_price,region
0,1,2024-01-01 00:00:00,1017.0,Electronics,3,44.68,West
1,2,2024-01-01 01:00:00,1154.0,Electronics,2,20.69,East
2,3,2024-01-01 02:00:00,1130.0,Apparel,4,41.60,West
3,4,2024-01-01 03:00:00,1087.0,Apparel,5,26.26,South
4,5,2024-01-01 04:00:00,1086.0,Apparel,2,39.45,West


In [7]:
orders.columns

Index(['order_id', 'order_date', 'customer_id', 'product_category', 'quantity',
       'unit_price', 'region'],
      dtype='str')

### Initial Observation

The generated dataset contains 5,015 rows and 7 columns. The additional 15 rows result from the intentionally introduced duplicate records.

At this stage, no values have been cleaned or modified. The dataset is preserved in its raw generated form so that all data-quality problems can be identified during the diagnosis stage.

In [8]:
assert orders.shape == (5015, 7)